# Iris classifier with Mage: blocks, tests and a quality gate

**Course:** Data Engineering - Pipeline Orchestration &nbsp;|&nbsp; **Assignment:** From Git to Gold, Part C

This notebook runs the `iris_classifier` pipeline of the Mage project in `iris_project/` and demonstrates the main idea of Part C:
**tests between blocks act as a gate** - if a result is not good enough, the pipeline stops before anything is saved.

### The pipeline
```
load_iris -> validate_iris -> prepare_data -> train_model -> save_results
                                                         \-> evaluate_model
```

| Block | Type | What it does | Its `@test` checks |
|---|---|---|---|
| `load_iris` | data_loader | Loads the 150-row Iris dataset | 150 rows and a `target` column |
| `validate_iris` (C4, own block) | transformer | Passes the data through | no missing values, exactly 3 classes |
| `prepare_data` | transformer | Adds a `split` column (80 % train / 20 % test, stratified) | 120 train rows and 30 test rows |
| `train_model` | transformer | Trains a decision tree; `max_depth` is a **runtime variable** | test accuracy >= 0.90 (the gate) |
| `save_results` | data_exporter | Writes `iris_predictions.csv` | the file exists |
| `evaluate_model` (C3) | transformer | Confusion matrix and per-class metrics | 3 class rows + accuracy row |

**Requirements:** `pip install -r requirements-mage.txt` (Python 3.12 recommended). Run the notebook with the repository root as working directory.

In [1]:
import os
if os.path.basename(os.getcwd()) == 'iris_project':
    os.chdir('..')      # run everything from the repository root
print('working directory:', os.getcwd())

working directory: /home/prince_deepak_siddharth/Documents/sem 7/freshcart-pipelines


## 1. The pipeline definition
`metadata.yaml` wires the blocks together. Note that `train_model` lists **two** downstream blocks (`save_results` and `evaluate_model`) - this is the evaluation branch of task C3.

In [2]:
!cat iris_project/pipelines/iris_classifier/metadata.yaml

name: iris_classifier
uuid: iris_classifier
type: python
blocks:
- uuid: load_iris
  name: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks:
  - validate_iris
- uuid: validate_iris
  name: validate_iris
  type: transformer
  language: python
  upstream_blocks:
  - load_iris
  downstream_blocks:
  - prepare_data
- uuid: prepare_data
  name: prepare_data
  type: transformer
  language: python
  upstream_blocks:
  - validate_iris
  downstream_blocks:
  - train_model
- uuid: train_model
  name: train_model
  type: transformer
  language: python
  upstream_blocks:
  - prepare_data
  downstream_blocks:
  - save_results
  - evaluate_model
- uuid: save_results
  name: save_results
  type: data_exporter
  language: python
  upstream_blocks:
  - train_model
  downstream_blocks: []
- uuid: evaluate_model
  name: evaluate_model
  type: transformer
  language: python
  upstream_blocks:
  - train_model
  downstream_blocks: []


## 2. C1 - Run the whole pipeline with its tests
`--test` makes Mage run each block's `@test` function straight after the block. Every block should report **1/1 tests passed**.
With the default `max_depth = 3` the test accuracy is about 0.967, so the pipeline completes, saves the CSV and prints the confusion matrix and metrics table of `evaluate_model`.

In [3]:
!rm -f iris_predictions.csv
!mage run iris_project iris_classifier --test 2>&1 | grep -vE 'traitlets|DBT|INFO:|FutureWarning'

--------------------------------------------------------------
1/1 tests passed.
validate_iris: 150 rows, 3 classes, 0 missing values
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
max_depth=3  test accuracy=0.9667
--------------------------------------------------------------
1/1 tests passed.
saved 30 predictions -> iris_predictions.csv
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix (rows = true, columns = predicted):
                 pred_setosa  pred_versicolor  pred_virginica
true_setosa               10                0               0
true_versicolor            0                9               1
true_virginica             0                0              10

Metrics:
     class  precision  recall    f1  support
    setosa      1.000     1.0 1.000       10
versicolor      1.000     0.9 0.947       10
 virginica      0

## 3. C2 - Tests as a gate
We now run the pipeline for `max_depth` = 1 to 5, passing the value at run time with `--runtime-vars`. Before each run the CSV is deleted, so we can see whether `save_results` ran.

**What to expect:** with `max_depth = 1` the tree is a single split and reaches only 0.667 accuracy. The `@test` in `train_model` fails, Mage stops the run, `save_results` never starts and **no CSV is created**. For depth 2 to 5 the accuracy is above 0.90, the test passes and the file is written.

In [4]:
import os, subprocess
for d in range(1, 6):
    if os.path.exists('iris_predictions.csv'): os.remove('iris_predictions.csv')
    r = subprocess.run(['mage','run','iris_project','iris_classifier','--runtime-vars',f'{{"max_depth": {d}}}','--test'], capture_output=True, text=True)
    out = r.stdout + r.stderr
    acc = [l.strip() for l in out.splitlines() if 'test accuracy' in l]
    print(f'{acc[0] if acc else "":<38} pipeline completed={"Pipeline run completed" in out!s:<6} iris_predictions.csv created={os.path.exists("iris_predictions.csv")}')

max_depth=1  test accuracy=0.6667      pipeline completed=False  iris_predictions.csv created=False


max_depth=2  test accuracy=0.9333      pipeline completed=True   iris_predictions.csv created=True


max_depth=3  test accuracy=0.9667      pipeline completed=True   iris_predictions.csv created=True


max_depth=4  test accuracy=0.9333      pipeline completed=True   iris_predictions.csv created=True


max_depth=5  test accuracy=0.9333      pipeline completed=True   iris_predictions.csv created=True


## 4. Summary
| max_depth | Test accuracy | Test passed? | CSV created? |
|---|---|---|---|
| 1 | 0.667 | No | No |
| 2 | 0.933 | Yes | Yes |
| 3 | 0.967 | Yes | Yes |
| 4 | 0.933 | Yes | Yes |
| 5 | 0.933 | Yes | Yes |

**Why a gate is better than checking afterwards:** a bad model is never saved, so nobody can pick up a poor result by mistake and there is nothing to clean up; the failure also appears at the exact block that caused it.